# BankScope JPM `sec2md` bake-off

Ovaj notebook izvršava ceo JPM-only eksperiment od čistog Colab runtime-a do rezultata:

1. klonira ili osvežava repo;
2. pravi projektni Python 3.13 `.venv` i u njega instalira projekat, pytest i Ruff;
3. pokreće testove i Ruff;
4. obezbeđuje fiksni JPM 2025 10-K i manifest;
5. pravi obe chunk varijante, Qwen embeddinge i zaključani retrieval bake-off;
6. pakuje tri rezultata za preuzimanje.

Pre početka izaberi **Runtime → Change runtime type → T4 GPU**, pa pokreni ćelije redom.

## 1. Parametri i pomoćna funkcija

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/nikolabakic/Banking-Technology-and-Operational-Risk-Intelligence-Assistant.git"
REPO_NAME = "Banking-Technology-and-Operational-Risk-Intelligence-Assistant"
BRANCH = "main"

CONTENT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
PROJECT_DIR = CONTENT_DIR / REPO_NAME
EXPERIMENT_DIR = PROJECT_DIR / "data/experiments/jpm_sec2md"

def run(command: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    print("$", " ".join(map(str, command)))
    subprocess.run(
        [str(item) for item in command],
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
    )

print(f"Project directory: {PROJECT_DIR}")

## 2. Repo i potrebni fajlovi

Ako repo već postoji, koristi se `git pull --ff-only`. Ako provera prijavi da fajl nedostaje, on još nije pushovan na izabrani branch.

In [ ]:
if (Path.cwd() / "pyproject.toml").exists() and (Path.cwd() / "src/bankscope").exists():
    PROJECT_DIR = Path.cwd().resolve()
    EXPERIMENT_DIR = PROJECT_DIR / "data/experiments/jpm_sec2md"
    print(f"Using current checkout: {PROJECT_DIR}")
elif (PROJECT_DIR / ".git").exists():
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT_DIR)])

run(["git", "log", "-1", "--oneline"], cwd=PROJECT_DIR)

required_paths = [
    "pyproject.toml",
    "tests/test_sec2md_adapter.py",
    "tests/test_hybrid_retriever.py",
    "src/bankscope/parsing/sec2md_adapter.py",
    "src/bankscope/retrieval/hybrid_retriever.py",
    "scripts/build_jpm_sec2md_experiment.py",
    "scripts/evaluate_jpm_sec2md_bakeoff.py",
    "scripts/generate_embeddings.py",
]
missing_paths = [path for path in required_paths if not (PROJECT_DIR / path).exists()]
for path in required_paths:
    print(f"{'NEDOSTAJE' if path in missing_paths else 'OK':10} {path}")
if missing_paths:
    raise FileNotFoundError("Pushuj nedostajuće fajlove na GitHub: " + ", ".join(missing_paths))

## 3. Projektni Python 3.13

Colabov kernel ostaje na 3.12, ali se ceo BankScope projekat izvršava kroz zaseban Python 3.13. Time se rešavaju i Python constraint i `/usr/bin/python3: No module named ruff`.

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"])
run([sys.executable, "-m", "uv", "python", "install", "3.13"])
run([
    sys.executable, "-m", "uv", "venv", "--python", "3.13", "--clear",
    str(PROJECT_DIR / ".venv"),
])

VENV_PYTHON = PROJECT_DIR / ".venv/bin/python"
if not VENV_PYTHON.exists():
    raise FileNotFoundError(f"Nije napravljen interpreter: {VENV_PYTHON}")

run([str(VENV_PYTHON), "--version"])
run([
    sys.executable, "-m", "uv", "pip", "install",
    "--python", str(VENV_PYTHON), "-e", ".[dev]",
], cwd=PROJECT_DIR)

## 4. Testovi i Ruff

In [ ]:
run([
    str(VENV_PYTHON), "-m", "pytest", "-q",
    "tests/test_sec2md_adapter.py", "tests/test_hybrid_retriever.py",
], cwd=PROJECT_DIR)

run([
    str(VENV_PYTHON), "-m", "ruff", "check",
    "src/bankscope/parsing/sec2md_adapter.py",
    "src/bankscope/retrieval/hybrid_retriever.py",
    "scripts/build_jpm_sec2md_experiment.py",
    "scripts/evaluate_jpm_sec2md_bakeoff.py",
], cwd=PROJECT_DIR)

## 5. Hugging Face i GPU

`HF_TOKEN` je opcion. Ako postoji u Colab Secrets, koristi se bez ispisivanja. Notebook prekida rad ako T4 nije uključen.

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN je učitan iz Colab Secrets.")
    else:
        print("HF_TOKEN nije podešen; modeli su javni pa nastavljam.")
except (ImportError, RuntimeError, KeyError):
    print("Colab Secrets nisu dostupni; nastavljam sa javnim modelima.")

RUN_ENV = os.environ.copy()
run([
    str(VENV_PYTHON), "-c",
    "import sys, torch; "
    "print('Python:', sys.version); "
    "print('Torch:', torch.__version__); "
    "print('CUDA:', torch.cuda.is_available()); "
    "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None); "
    "assert torch.cuda.is_available(), 'Uključi T4 GPU u Colab Runtime settings.'",
], cwd=PROJECT_DIR, env=RUN_ENV)

## 6. Fiksni JPM 2025 10-K

Eksperiment koristi isti filing kao ranije, ne promenljivi „najnoviji” 10-K. Postojeći HTML se ne preuzima ponovo.

In [ ]:
import json
from urllib.request import Request, urlopen

JPM_RECORD = {
    "ticker": "JPM",
    "cik": "0000019617",
    "legal_name": "JPMorgan Chase & Co.",
    "form": "10-K",
    "accession_number": "0001628280-26-008131",
    "filing_date": "2026-02-13",
    "report_date": "2025-12-31",
    "primary_document": "jpm-20251231.htm",
    "source_url": "https://www.sec.gov/Archives/edgar/data/19617/000162828026008131/jpm-20251231.htm",
    "local_html_path": "data/raw/sec/0000019617/000162828026008131/jpm-20251231.htm",
}

MANIFEST_PATH = PROJECT_DIR / "artifacts/manifests/filings.json"
RAW_HTML_PATH = PROJECT_DIR / JPM_RECORD["local_html_path"]

if not RAW_HTML_PATH.exists():
    RAW_HTML_PATH.parent.mkdir(parents=True, exist_ok=True)
    request = Request(
        JPM_RECORD["source_url"],
        headers={
            "User-Agent": "BankScope academic research nikolabakic@users.noreply.github.com",
            "Accept": "text/html,*/*",
        },
    )
    print("Preuzimam JPM 2025 10-K sa SEC EDGAR-a...")
    with urlopen(request, timeout=120) as response:
        RAW_HTML_PATH.write_bytes(response.read())
else:
    print("JPM HTML već postoji.")

if MANIFEST_PATH.exists():
    manifest_records = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    manifest_records = [row for row in manifest_records if row.get("ticker") != "JPM"]
else:
    manifest_records = []
manifest_records.append(JPM_RECORD)
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(
    json.dumps(manifest_records, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

html_size_mb = RAW_HTML_PATH.stat().st_size / (1024 * 1024)
print(f"Raw HTML: {RAW_HTML_PATH}")
print(f"Size: {html_size_mb:.2f} MiB")
print(f"Manifest: {MANIFEST_PATH}")
if html_size_mb < 1:
    raise ValueError("SEC filing je neočekivano mali; proveri odgovor servera.")

## 7. Izgradnja obe chunk varijante

In [ ]:
run([
    str(VENV_PYTHON), "scripts/build_jpm_sec2md_experiment.py",
    "--manifest", str(MANIFEST_PATH),
    "--raw-html", str(RAW_HTML_PATH),
    "--output-dir", str(EXPERIMENT_DIR),
    "--overwrite",
], cwd=PROJECT_DIR, env=RUN_ENV)

## 8. Qwen embedding: `sec2md_builtin`

In [ ]:
BUILTIN_DIR = EXPERIMENT_DIR / "sec2md_builtin"
run([
    str(VENV_PYTHON), "scripts/generate_embeddings.py",
    "--input", str(BUILTIN_DIR / "embedding_records.jsonl"),
    "--output", str(BUILTIN_DIR / "embeddings.npz"),
    "--checkpoint-dir", str(BUILTIN_DIR / "checkpoints"),
    "--batch-size", "8",
], cwd=PROJECT_DIR, env=RUN_ENV)

## 9. Qwen embedding: `structure_aware`

In [ ]:
STRUCTURE_DIR = EXPERIMENT_DIR / "structure_aware"
run([
    str(VENV_PYTHON), "scripts/generate_embeddings.py",
    "--input", str(STRUCTURE_DIR / "embedding_records.jsonl"),
    "--output", str(STRUCTURE_DIR / "embeddings.npz"),
    "--checkpoint-dir", str(STRUCTURE_DIR / "checkpoints"),
    "--batch-size", "8",
], cwd=PROJECT_DIR, env=RUN_ENV)

## 10. Retrieval bake-off sa stvarnih 30 kandidata

In [ ]:
run([
    str(VENV_PYTHON), "scripts/evaluate_jpm_sec2md_bakeoff.py",
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--candidate-k", "30",
    "--rrf-k", "60",
    "--reranker-batch-size", "4",
    "--reranker-device", "cuda",
], cwd=PROJECT_DIR, env=RUN_ENV)

## 11. Pregled i preuzimanje rezultata

In [ ]:
import shutil

RESULT_PATH = EXPERIMENT_DIR / "retrieval_bakeoff.json"
results = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
print(json.dumps(results.get("summary", results), ensure_ascii=False, indent=2)[:12000])

result_files = [
    EXPERIMENT_DIR / "experiment_manifest.json",
    EXPERIMENT_DIR / "qrels_audit.json",
    RESULT_PATH,
]
missing_results = [str(path) for path in result_files if not path.exists()]
if missing_results:
    raise FileNotFoundError("Nedostaju rezultati: " + ", ".join(missing_results))

bundle_dir = PROJECT_DIR / "bankscope_jpm_sec2md_results"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True)
for source_path in result_files:
    shutil.copy2(source_path, bundle_dir / source_path.name)

zip_path = Path(shutil.make_archive(str(bundle_dir), "zip", root_dir=bundle_dir))
print(f"Spremno za preuzimanje: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Van Colaba: preuzmi ZIP sa prikazane putanje.")